# 04 — Virtual metrology

PR curve, confusion matrix, threshold.

In [ ]:
from fabsentinel.vm.train import load
artifact = load('artifacts/vm_model.joblib')
print('Test metrics:')
for k, v in artifact.metrics.items():
    print(f'  {k}: {v}')
print('\nCV metrics:')
for k, v in artifact.cv_metrics.items():
    print(f'  {k}: {v}')


In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from fabsentinel.vm.features import prepare
from fabsentinel.vm.evaluate import pr_curve
from sklearn.model_selection import train_test_split
df = pd.read_parquet('data/cleaned.parquet')
sigcols = [c for c in df.columns if c.startswith('signal_')]
X, y = prepare(df, sigcols)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
proba = artifact.model.predict_proba(X_test)[:, 1]
pr = pr_curve(y_test, proba)
plt.figure(figsize=(6, 5))
plt.plot(pr['recall'], pr['precision'], color='#dd6b20')
plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title(f'PR curve (PR-AUC = {artifact.metrics["test_pr_auc"]:.3f})')
plt.axvline(x=pr.loc[pr['threshold'].sub(artifact.threshold).abs().idxmin(), 'recall'], color='gray', ls='--', alpha=0.5)
plt.show()
